# Бейзлайн Gate 4–5

Данные учебные и сгенерированы. Ноутбук показывает два независимых замера: модель на размеченном корпусе и модели на панели против правил Gate 3. Их метрики не сравниваются между собой.

In [1]:
import json, sys
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
from labeling import agreement, baseline, expand, heuristic, score, LAB
pd.set_option("display.max_colwidth", 90, "display.width", 140)

## 1. Размеченный корпус

277 уникальных сгенерированных случаев. Размечал Claude по гайдлайну; второй проход — 70 случаев.

In [2]:
data = pd.read_csv(LAB / "labeled_dataset.csv")
print(len(data), data.text.is_unique)
for col in ["outcome", "reason", "manageable", "action"]:
    print(data[col].value_counts().to_dict())
data.sample(5, random_state=1)[["id", "text", "outcome", "reason", "manageable", "action"]]

277 True
{'left': 208, 'unknown': 38, 'stayed': 31}
{'price_fixed': 51, 'price_fee': 50, 'unknown': 43, 'forced': 40, 'service': 37, 'credit_counterparty': 31, 'other': 25}
{'no': 140, 'yes': 94, 'na': 43}
{'none': 183, 'package': 45, 'lower_tariff': 42, 'annual': 7}


,id,text,outcome,reason,manageable,action
276,GEN-277,"Боюсь блокировки, поэтому перевёл деньги в другой банк.",left,other,no,none
88,GEN-089,"Переехал в другой регион, там у банка нет отделений для инкассации. Счёт закрыл.",left,other,no,none
95,GEN-096,"Комиссия за перевод возмутила, но вернуться не смогу: уже всё перевёл, поздно предлагать.",left,price_fee,no,none
258,GEN-259,"Торговая сеть принимает оплату только на счета партнёрского банка, расчёты перенесены.",left,credit_counterparty,no,none
12,GEN-013,"Банк отказал в обслуживании, мне пришлось уходить, хотя условия устраивали.",left,forced,no,none


## 2. Согласованность проходов

In [3]:
first, second = expand(LAB / "labels_round1_compact.txt"), expand(LAB / "labels_round2_compact.txt")
agree, merged, disagreements = agreement(first, second)
display(pd.DataFrame(agree).T[["kappa", "agreement", "n", "ci95"]])
disagreements[["id", "outcome_1", "outcome_2", "reason_1", "reason_2", "manageable_1", "manageable_2"]]

,kappa,agreement,n,ci95
outcome,0.901206,0.957746,71,"[0.774838, 1.0]"
reason,0.9669,0.971831,71,"[0.916431, 1.0]"
manageable,0.977453,0.985915,71,"[0.927473, 1.0]"
action,0.971243,0.985915,71,"[0.89784, 1.0]"


,id,outcome_1,outcome_2,reason_1,reason_2,manageable_1,manageable_2
19,GEN-072,left,left,credit_counterparty,other,no,no
20,GEN-078,left,left,credit_counterparty,other,no,no
32,GEN-136,left,left,price_fixed,price_fixed,yes,no
33,GEN-139,left,unknown,other,other,no,no
56,GEN-223,left,unknown,other,other,no,no
61,GEN-232,unknown,stayed,price_fixed,price_fixed,yes,yes


## 3. Бейзлайн на размеченных данных

Цель — управляемая причина. Отложенная часть позже по дате.

In [4]:
result, errors = baseline(data)
display(pd.DataFrame({k: result["holdout"][k] for k in ["model", "keywords"]}).T)
print({k: result["holdout"][k] for k in ["split_month", "train", "test", "positive_test", "f1_delta_ci95"]})
errors[["text", "manageable", "predicted", "reason"]].head(8)

,precision,recall,f1
model,0.774194,0.827586,0.800000
keywords,0.758621,0.758621,0.758621


{'split_month': '2025-06', 'train': 162, 'test': 72, 'positive_test': 29, 'f1_delta_ci95': [-0.08804130897351245, 0.16843219461697703]}


,text,manageable,predicted,reason
9,"Смс с кодами приходят через раз, платить невозможно. Ушёл.",no,yes,service
22,"Комиссии высокие, поддержка слабая, но решающим стал сбой в день зарплаты. Закрыл счёт.",no,yes,service
23,Основной контрагент сменил банк и попросил нас открыть счёт там же. Платежи перевели.,no,yes,credit_counterparty
24,"Сделали пакет платежей, но после трёх бесплатных месяцев стал дороже, чем без него. Ушли.",no,yes,price_fee
27,"Снятие наличных подорожало, для моего магазина это критично. Теперь работаю через друг...",yes,no,price_fee
29,"Зарплатные выплаты самозанятым обходятся в 1,9% — это грабёж. Сменил банк в тот же месяц.",yes,no,price_fee
33,Банк приравнял мой перевод к снятию наличных и взял 2%. После этого вывел все деньги и...,yes,no,price_fee
53,"Плата выросла, зато поддержка отличная, поэтому не ухожу.",yes,no,price_fixed


## 4. Панель: модели против правил Gate 3

Тест: 12 месяцев 2025 года. Истинный уход — скрытый эталон симулятора, в обучении не участвует.

In [5]:
main = json.loads((ROOT / "results" / "main.json").read_text())
names = ["rules", "logistic_v2", "catboost_g3_features", "catboost_v2", "catboost_v2s"]
for title, key in [("Официальный таргет", "official_target"), ("Истинный уход за 60 дней", "true_churn60")]:
    print(title)
    display(pd.DataFrame(main["risk"][key]).T.loc[names, ["positive", "base_rate", "precision", "lift", "recall", "brier"]].astype(float).round(4))

Официальный таргет


,positive,base_rate,precision,lift,recall,brier
rules,7164.0,0.0295,0.0669,2.2665,0.2267,0.0286
logistic_v2,7164.0,0.0295,0.1266,4.2901,0.4291,0.0286
catboost_g3_features,7164.0,0.0295,0.1231,4.1729,0.4174,0.0267
catboost_v2,7164.0,0.0295,0.1373,4.6530,0.4654,0.0229
catboost_v2s,7164.0,0.0295,0.0662,2.2442,0.2245,0.0285


Истинный уход за 60 дней


,positive,base_rate,precision,lift,recall,brier
rules,4530.0,0.0187,0.0248,1.3309,0.1331,0.0183
logistic_v2,4530.0,0.0187,0.0376,2.0151,0.2015,0.0329
catboost_g3_features,4530.0,0.0187,0.0449,2.4080,0.2408,0.0253
catboost_v2,4530.0,0.0187,0.0479,2.5669,0.2567,0.0268
catboost_v2s,4530.0,0.0187,0.0559,2.9973,0.2998,0.0181


## 5. Честность замера

In [6]:
print("Границы периодов валидны:", main["split_cuts_valid"])
print("Мутация будущего не меняет прогноз:", main["future_mutation_invariant"])
print("Шум метки:", {k: round(v, 3) for k, v in main["label_noise"].items()})
print("Важность признаков, тихий отток:")
pd.Series(main["importance_quiet"]).round(1).head(10)

Границы периодов валидны: True
Мутация будущего не меняет прогноз: True
Шум метки: {'quiet_positive': 7164, 'quiet_no_true_churn_share': 0.368, 'quiet_hibernator_share': 0.383, 'adj_removed': 2562, 'true_churn_missed_by_label_share': 0.0, 'hidden_forced_in_positive_share': 0.104}
Важность признаков, тихий отток:


self_transfer_share        15.4
fee_shock                  12.9
fee_burden                 11.0
season_adjusted_ratio      10.5
yoy_ratio_3m                9.2
services                    8.0
yoy_ratio                   6.0
complaints                  5.0
last_year_forward_ratio     4.7
cash_buffer                 4.3
dtype: float64

## 6. Эффект и экономика

Глубина 10%, отбор моделью v2s, истинный эффект по скрытым потенциальным исходам.

In [7]:
rows = pd.DataFrame(main["policies"]["0.1"]).T
rows[["selected", "uplift12_pp", "uplift60_pp", "mean_cost", "breakeven_pp", "revenue_per_budget", "after_ops_per_1000"]].astype(float).round(2)

,selected,uplift12_pp,uplift60_pp,mean_cost,breakeven_pp,revenue_per_budget,after_ops_per_1000
rules,24289.0,0.05,-0.07,6028.49,20.71,0.00,-5563467.45
risk,24289.0,0.78,0.33,5995.66,20.55,0.08,-5336840.90
risk_adj,24289.0,0.91,0.30,5982.60,20.47,0.10,-5277594.79
risk_x_revenue,24289.0,0.74,0.23,6093.21,14.85,0.08,-5409231.86
random,24289.0,-0.03,-0.07,6047.98,20.69,-0.00,-5598690.04
uplift_x_revenue,2333.0,0.47,0.60,1943.63,5.00,0.28,-1653617.33
oracle_bound,1104.0,100.00,10.87,4696.15,16.22,12.77,22604348.16


## 7. Эвалы

In [8]:
evals = pd.read_csv(ROOT / "eval" / "eval_set.csv")
print(evals.groupby("severity").passed.agg(["sum", "count"]))
evals[~evals.passed][["id", "severity", "scenario", "expected", "observed"]]

          sum  count
severity            
critical   21     21
major      12     17


,id,severity,scenario,expected,observed
18,E18,major,C4.4 Разрыв recall по отраслям,≤ 5 п.п.,5.58; p=0.14
31,E31,major,Кампания на 10% покрывает ≥ 30% оттока,"Lift@10% ≥ 3,0",2.997
32,E32,major,Есть время на контакт,Медианное упреждение ≥ 30 дней,26.8
33,E33,major,Эффект окупает оффер,12-мес. uplift ≥ безубыточности,0.91 / 20.47
34,E34,major,Метрика успеха продукта,≥ 3 ₽ на 1 ₽ бюджета,0.1
